# Public relay power curves

## Init

In [ ]:
from pathlib import Path
import io
import subprocess
import tarfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.optimize import curve_fit
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.dpi': 110,
    'savefig.dpi': 200,
    'font.size': 11,
    'axes.titlesize': 12,
    'axes.labelsize': 11,
})

REPO = Path(subprocess.run(
    ['git', 'rev-parse', '--show-toplevel'], check=True, capture_output=True, text=True
).stdout.strip())
FIG_DIR = REPO / 'analysis' / 'figures' / 'section3_open_relays'
FIG_DIR.mkdir(parents=True, exist_ok=True)

COLORS = {
    'France': '#377eb8',
    'Germany': '#e41a1c',
    'Great Britain': '#4daf4a',
}
RELAY_ORDER = list(COLORS)

ANALYSIS_WINDOWS = {
    'France': (pd.Timestamp('2025-08-25', tz='UTC'), pd.Timestamp('2026-05-06', tz='UTC')),
    'Germany': (pd.Timestamp('2025-08-25', tz='UTC'), pd.Timestamp('2026-05-06', tz='UTC')),
    'Great Britain': (pd.Timestamp('2025-11-08', tz='UTC'), pd.Timestamp('2026-05-07', tz='UTC')),
}
PAPER_POWER_MODELS = {
    'France': (3.8866, 8.8392),
    'Great Britain': (6.7951, -11.5217),
}
LOW_THROUGHPUT_MAX = 1.0

print(f'Repository: {REPO}')
print(f'Figures: {FIG_DIR}')

## Load the public traces

In [ ]:
ARCHIVES = {
    'France': 'france/OF-relay-1min-avg-08-01-merged.csv.tar.gz',
    'Germany': 'germany/OG-relay-1min-avg-08-01-merged.csv.tar.gz',
    'Great Britain': 'great-britain/OBR-relay-1min-avg-08-01-merged.csv.tar.gz',
}
HARDWARE = {
    'France': 'AMD Ryzen 5 3600X',
    'Germany': 'AMD Ryzen 5 3600',
    'Great Britain': 'Intel Xeon E-2236',
}

frames = []
for relay, relative_path in ARCHIVES.items():
    archive_path = (REPO / relative_path).resolve()
    with tarfile.open(archive_path, mode='r:*') as archive:
        csv_member = next(member for member in archive.getmembers() if member.name.endswith('.csv'))
        with archive.extractfile(csv_member) as csv_file:
            relay_df = pd.read_csv(csv_file)

    relay_df['timestamp'] = pd.to_datetime(relay_df['timestamp'], utc=True, errors='coerce')
    relay_df['relay'] = relay
    relay_df['hardware'] = HARDWARE[relay]
    frames.append(relay_df)

master_df = pd.concat(frames, ignore_index=True)
numeric_columns = ['mbps', 'circuits', 'power', 'cpu', 'packets', 'avg_pkt_size']
master_df[numeric_columns] = master_df[numeric_columns].apply(pd.to_numeric, errors='coerce')
negative_sentinels = master_df[numeric_columns].lt(0)
sentinel_summary = (
    negative_sentinels.assign(relay=master_df['relay'])
    .groupby('relay')[numeric_columns].sum().astype(int)
    .reindex(RELAY_ORDER)
)
master_df[numeric_columns] = master_df[numeric_columns].mask(negative_sentinels)
master_df = (
    master_df.dropna(subset=['timestamp', 'relay'])
             .sort_values(['relay', 'timestamp'])
             .reset_index(drop=True)
)
master_df['date'] = master_df['timestamp'].dt.date
master_df['month'] = master_df['timestamp'].dt.strftime('%Y-%m')
master_df['architecture'] = np.where(master_df['relay'].eq('Great Britain'), 'Intel', 'AMD')

print(f'Master rows: {len(master_df):,}')
display(master_df.head())

display(sentinel_summary)

## Coverage

The analysis uses the power-model data windows: August 25, 2025 through May 5, 2026 for France and Germany, and November 8, 2025 through May 6, 2026 for Great Britain. End dates are inclusive.

In [ ]:
coverage = (
    master_df.groupby('relay', observed=True)
             .agg(
                 first_timestamp=('timestamp', 'min'),
                 last_timestamp=('timestamp', 'max'),
                 rows=('timestamp', 'size'),
                 duplicate_timestamps=('timestamp', lambda x: x.duplicated().sum()),
             )
             .reindex(RELAY_ORDER)
)
coverage['span_days'] = (coverage['last_timestamp'] - coverage['first_timestamp']).dt.total_seconds() / 86400
display(coverage)

window_parts = []
for relay, (start, end_exclusive) in ANALYSIS_WINDOWS.items():
    relay_window = master_df[
        master_df['relay'].eq(relay)
        & master_df['timestamp'].ge(start)
        & master_df['timestamp'].lt(end_exclusive)
    ].copy()
    window_parts.append(relay_window)

curve_df = pd.concat(window_parts, ignore_index=True)
window_counts = (
    curve_df.groupby('relay').size().rename('analysis rows')
            .reindex(RELAY_ORDER).astype('Int64').to_frame()
)
window_counts['start'] = [ANALYSIS_WINDOWS[relay][0] for relay in RELAY_ORDER]
window_counts['end'] = [ANALYSIS_WINDOWS[relay][1] - pd.Timedelta(days=1) for relay in RELAY_ORDER]
display(window_counts)

missing_value_summary = (
    curve_df.groupby('relay')[numeric_columns]
    .agg(lambda values: values.isna().sum())
    .astype(int)
    .reindex(RELAY_ORDER)
)
display(missing_value_summary)

## Independent variables

The curves below are marginal relationships: each variable is plotted against power without holding the others fixed. Negative values are missing-value sentinels and are converted to `NaN`; genuine zeros are retained. Each curve and correlation uses rows where that feature and power are both observed. CPU utilization is included as a measurement diagnostic, but it is not an input available for the Section 5 Tor Metrics estimator.

In [ ]:
PREDICTORS = {
    'mbps': {'label': 'Throughput (Mbps)', 'scale': 'linear'},
    'circuits': {'label': 'Circuit count', 'scale': 'linear'},
    'cpu': {'label': 'CPU utilization', 'scale': 'linear'},
    'packets': {'label': 'Packet count', 'scale': 'linear'},
    'avg_pkt_size': {'label': 'Average packet size (bytes)', 'scale': 'linear'},
}

summary_rows = []
for relay in RELAY_ORDER:
    relay_data = curve_df[curve_df['relay'].eq(relay)]
    for variable, meta in PREDICTORS.items():
        values = relay_data[variable].dropna()
        summary_rows.append({
            'relay': relay,
            'variable': variable,
            'p01': values.quantile(0.01),
            'median': values.median(),
            'p99': values.quantile(0.99),
            'zero_pct': 100 * values.eq(0).mean(),
        })
variable_summary = pd.DataFrame(summary_rows)
display(variable_summary.round(3))

correlation_rows = []
for relay in RELAY_ORDER:
    relay_data = curve_df[curve_df['relay'].eq(relay)]
    for variable in PREDICTORS:
        pair = relay_data[[variable, 'power']].dropna()
        correlation_rows.append({
            'relay': relay,
            'variable': variable,
            'valid_pair_rows': len(pair),
            'spearman_rho': pair[variable].corr(pair['power'], method='spearman'),
        })
correlation_df = pd.DataFrame(correlation_rows)
correlation_matrix = correlation_df.pivot(index='variable', columns='relay', values='spearman_rho')
correlation_matrix = correlation_matrix.reindex(index=PREDICTORS, columns=RELAY_ORDER)
display(correlation_matrix.round(3))

## Marginal power curves

Each curve summarizes 60 equal-frequency bins over positive predictor values. Lines are median power and bands are the interquartile range. Faint points are a reproducible sample. In the throughput panels, every observation at or below 1 Mbps is also included as a raw point so rare near-idle intervals cannot disappear inside an equal-frequency bin.

In [ ]:
binned_rows = []
for relay in RELAY_ORDER:
    relay_data = curve_df[curve_df['relay'].eq(relay)]
    for variable in PREDICTORS:
        pair = relay_data[[variable, 'power']].replace([np.inf, -np.inf], np.nan).dropna()
        pair = pair[(pair[variable] > 0) & (pair['power'] > 0)].copy()
        pair['bin'] = pd.qcut(pair[variable], q=60, labels=False, duplicates='drop')
        curve = (
            pair.groupby('bin', observed=True)
                .agg(
                    x=(variable, 'median'),
                    power_q25=('power', lambda x: x.quantile(0.25)),
                    power_median=('power', 'median'),
                    power_q75=('power', lambda x: x.quantile(0.75)),
                    n=('power', 'size'),
                )
                .reset_index(drop=True)
        )
        curve['relay'] = relay
        curve['variable'] = variable
        binned_rows.append(curve)

binned_curves = pd.concat(binned_rows, ignore_index=True)
display(binned_curves.head())

## Paper figures

These figures use the large-font style from the Section 5 notebook. Feature plots overlay all three relays; the solid line is the binned median trend and the legend reports Spearman's correlation. The dotted throughput curves reproduce the natural-log regression pipeline in `power-models/log-regression.py`.

In [ ]:
import matplotlib.dates as mdates
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

PAPER_FIG_DIR = FIG_DIR / 'paper'
PAPER_FIG_DIR.mkdir(parents=True, exist_ok=True)

PAPER_FONT_SIZE = 22
PAPER_LABEL_SIZE = 24
PAPER_LEGEND_SIZE = 18
PAPER_LINE_WIDTH = 3.0
PAPER_MARKER_SIZE = 8
PAPER_RELAY_MARKERS = {
    'France': 'o',
    'Germany': 's',
    'Great Britain': '^',
}
PAPER_FEATURE_STEMS = {
    'mbps': 'throughput',
    'circuits': 'circuits',
    'cpu': 'cpu',
    'packets': 'packets',
    'avg_pkt_size': 'average_packet_size',
}
POWER_COLOR = '#D62728'
THROUGHPUT_COLOR = '#1F77B4'
PAPER_POWER_MAX = curve_df['power'].quantile(0.999) * 1.08

def format_paper_axes(ax):
    ax.grid(axis='both', linestyle='--', alpha=0.25)
    ax.tick_params(axis='both', labelsize=PAPER_FONT_SIZE, direction='in')
    for spine in ax.spines.values():
        spine.set_color('gray')
        spine.set_linewidth(0.6)

def save_paper_figure(fig, stem):
    png_path = PAPER_FIG_DIR / f'{stem}.png'
    pdf_path = PAPER_FIG_DIR / f'{stem}.pdf'
    fig.savefig(png_path, bbox_inches='tight', pad_inches=0.05, dpi=300)
    fig.savefig(pdf_path, bbox_inches='tight', pad_inches=0.05)
    return png_path, pdf_path

print(PAPER_FIG_DIR)

### Longitudinal appendix figures

These use daily medians and interquartile ranges over each relay's power-model window. Gray spans mark days with no collected observations.

In [ ]:
daily_timeseries_df = (
    curve_df.assign(day=curve_df['timestamp'].dt.floor('D'))
    .groupby(['relay', 'day'], as_index=False, observed=True)
    .agg(
        power_q25=('power', lambda x: x.quantile(0.25)),
        power_median=('power', 'median'),
        power_q75=('power', lambda x: x.quantile(0.75)),
        throughput_q25=('mbps', lambda x: x.quantile(0.25)),
        throughput_median=('mbps', 'median'),
        throughput_q75=('mbps', lambda x: x.quantile(0.75)),
        minutes=('timestamp', 'size'),
    )
)
display(daily_timeseries_df.head())

In [ ]:

PAPER_FONT_SIZE = 26
PAPER_LABEL_SIZE = 26
PAPER_LEGEND_SIZE = 24
PAPER_LINE_WIDTH = 3.0
PAPER_MARKER_SIZE = 8

In [ ]:
longitudinal_figure_rows = []

for relay in RELAY_ORDER:
    relay_daily = daily_timeseries_df[daily_timeseries_df['relay'].eq(relay)].copy()
    start, end_exclusive = ANALYSIS_WINDOWS[relay]
    full_days = pd.date_range(
        start.normalize(), end_exclusive.normalize() - pd.Timedelta(days=1), freq='D'
    )
    relay_daily = relay_daily.set_index('day').reindex(full_days)
    no_data = relay_daily['minutes'].isna()
    gap_groups = no_data.ne(no_data.shift(fill_value=False)).cumsum()

    fig, axes = plt.subplots(2, 1, figsize=(9, 5.4), sharex=True)
    mark_every = max(1, len(relay_daily) // 14)

    axes[0].fill_between(
        relay_daily.index, relay_daily['power_q25'], relay_daily['power_q75'],
        color=POWER_COLOR, alpha=0.14, linewidth=0,
    )
    axes[0].plot(
        relay_daily.index, relay_daily['power_median'],
        color=POWER_COLOR, linewidth=PAPER_LINE_WIDTH,
        marker='o', markersize=4, markevery=mark_every,
    )
    axes[1].fill_between(
        relay_daily.index, relay_daily['throughput_q25'], relay_daily['throughput_q75'],
        color=THROUGHPUT_COLOR, alpha=0.14, linewidth=0,
    )
    axes[1].plot(
        relay_daily.index, relay_daily['throughput_median'],
        color=THROUGHPUT_COLOR, linewidth=PAPER_LINE_WIDTH,
        marker='s', markersize=4, markevery=mark_every,
    )

    for _, gap in relay_daily[no_data].groupby(gap_groups[no_data]):
        gap_start = gap.index.min()
        gap_end = gap.index.max() + pd.Timedelta(days=1)
        for ax in axes:
            ax.axvspan(gap_start, gap_end, color='0.78', alpha=0.55, linewidth=0, zorder=0)

    axes[0].set_title(relay, fontsize=PAPER_LABEL_SIZE, pad=12)
    axes[0].set_ylabel('Power (W)', fontsize=PAPER_LABEL_SIZE, labelpad=5)
    axes[1].set_ylabel('Throughput (Mbps)', fontsize=PAPER_LABEL_SIZE, labelpad=20)
    axes[1].set_xlabel('Date (UTC)', fontsize=PAPER_LABEL_SIZE, labelpad=10)
    axes[0].set_ylim(bottom=0)
    axes[1].set_ylim(bottom=0)
    axes[1].xaxis.set_major_locator(mdates.MonthLocator(interval=2))
    axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%b\n%Y'))
    axes[1].set_xlim(start, end_exclusive - pd.Timedelta(days=1))
    axes[0].legend(
        handles=[Patch(facecolor='0.78', alpha=0.55, label='No data collected')],
        loc='best', frameon=True, fontsize=PAPER_LEGEND_SIZE,
    )
    for ax in axes:
        format_paper_axes(ax)
    fig.subplots_adjust(left=0.16, right=0.98, bottom=0.18, top=0.92, hspace=0.18)

    relay_stem = relay.lower().replace(' ', '_')
    stem = f'paper_{relay_stem}_power_throughput_over_time'
    png_path, pdf_path = save_paper_figure(fig, stem)
    longitudinal_figure_rows.append({
        'relay': relay, 'png': png_path, 'pdf': pdf_path,
    })
    plt.show()

longitudinal_figures = pd.DataFrame(longitudinal_figure_rows)
display(longitudinal_figures)

### Time of day

Each date-hour is averaged first, then those values are averaged across days. Bands show 95% confidence intervals across daily averages. Hours are UTC.

In [ ]:
daily_hour_df = (
    curve_df.assign(
        day=curve_df['timestamp'].dt.floor('D'),
        hour=curve_df['timestamp'].dt.hour,
    )
    .groupby(['relay', 'day', 'hour'], as_index=False, observed=True)
    .agg(power=('power', 'mean'), throughput=('mbps', 'mean'))
)

time_of_day_df = (
    daily_hour_df.groupby(['relay', 'hour'], as_index=False, observed=True)
    .agg(
        power_mean=('power', 'mean'),
        power_std=('power', 'std'),
        throughput_mean=('throughput', 'mean'),
        throughput_std=('throughput', 'std'),
        days=('day', 'nunique'),
    )
)
time_of_day_df['power_ci95'] = (
    1.96 * time_of_day_df['power_std'].fillna(0) / np.sqrt(time_of_day_df['days'])
)
time_of_day_df['throughput_ci95'] = (
    1.96 * time_of_day_df['throughput_std'].fillna(0) / np.sqrt(time_of_day_df['days'])
)
display(time_of_day_df.head())

In [ ]:

PAPER_FONT_SIZE = 42
PAPER_LABEL_SIZE = 44
PAPER_LEGEND_SIZE = 42
PAPER_LINE_WIDTH = 3.0
PAPER_MARKER_SIZE = 8


In [ ]:
fig, power_axes = plt.subplots(1, 3, figsize=(19, 6.2), sharex=True)

for ax, relay in zip(power_axes, RELAY_ORDER):
    relay_hour = time_of_day_df[time_of_day_df['relay'].eq(relay)].sort_values('hour')
    throughput_ax = ax.twinx()

    power_lower = np.maximum(0, relay_hour['power_mean'] - relay_hour['power_ci95'])
    power_upper = relay_hour['power_mean'] + relay_hour['power_ci95']
    throughput_lower = np.maximum(
        0, relay_hour['throughput_mean'] - relay_hour['throughput_ci95']
    )
    throughput_upper = relay_hour['throughput_mean'] + relay_hour['throughput_ci95']

    ax.fill_between(
        relay_hour['hour'], power_lower, power_upper,
        color=POWER_COLOR, alpha=0.14, linewidth=0,
    )
    ax.plot(
        relay_hour['hour'], relay_hour['power_mean'],
        color=POWER_COLOR, linewidth=PAPER_LINE_WIDTH,
        marker='o', markersize=6, markevery=2,
    )
    throughput_ax.fill_between(
        relay_hour['hour'], throughput_lower, throughput_upper,
        color=THROUGHPUT_COLOR, alpha=0.12, linewidth=0,
    )
    throughput_ax.plot(
        relay_hour['hour'], relay_hour['throughput_mean'],
        color=THROUGHPUT_COLOR, linewidth=PAPER_LINE_WIDTH,
        marker='s', markersize=6, markevery=2,
    )

    ax.set_title(relay, fontsize=PAPER_LABEL_SIZE, pad=12)
    ax.set_xlabel('Hour (UTC)', fontsize=PAPER_LABEL_SIZE, labelpad=10)
    if relay == RELAY_ORDER[0]:
        ax.set_ylabel('Power (W)', fontsize=PAPER_LABEL_SIZE, color=POWER_COLOR, labelpad=8)
    if relay == RELAY_ORDER[-1]:
        throughput_ax.set_ylabel(
            'Throughput (Mbps)', fontsize=PAPER_LABEL_SIZE,
            color=THROUGHPUT_COLOR, labelpad=8,
        )
    ax.set_xlim(-0.3, 23.3)
    ax.set_xticks(range(0, 24, 4))
    power_min = float(power_lower.min())
    power_max = float(power_upper.max())
    power_padding = max(0.25, 0.12 * (power_max - power_min))
    ax.set_ylim(max(0, power_min - power_padding), power_max + power_padding)
    throughput_ax.set_ylim(bottom=0)
    format_paper_axes(ax)
    throughput_ax.tick_params(
        axis='y', labelsize=PAPER_FONT_SIZE, colors=THROUGHPUT_COLOR, direction='in'
    )
    throughput_ax.spines['right'].set_color('gray')
    throughput_ax.spines['right'].set_linewidth(0.6)

metric_handles = [
    Line2D([0], [0], color=POWER_COLOR, marker='o',
           linewidth=PAPER_LINE_WIDTH, markersize=7, label='Power'),
    Line2D([0], [0], color=THROUGHPUT_COLOR, marker='s',
           linewidth=PAPER_LINE_WIDTH, markersize=7, label='Throughput'),
]
fig.legend(
    handles=metric_handles, ncol=2, loc='upper center',
    fontsize=PAPER_LEGEND_SIZE, frameon=True, bbox_to_anchor=(0.5, 1.1),
)
fig.subplots_adjust(left=0.06, right=1.1, bottom=0.18, top=0.78, wspace=0.42)
png_path, pdf_path = save_paper_figure(fig, 'paper_power_throughput_by_time_of_day')
print(png_path)
print(pdf_path)
plt.show()

In [ ]:
PAPER_FONT_SIZE = 32
PAPER_LABEL_SIZE = 32
PAPER_LEGEND_SIZE = 18
PAPER_LINE_WIDTH = 3.0
PAPER_MARKER_SIZE = 8

In [ ]:
pooled = (
    curve_df[['mbps', 'power']]
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
)
pooled = pooled[pooled['mbps'] > 0]
x_high = pooled['mbps'].quantile(0.999)

fig, ax = plt.subplots(figsize=(9.2, 5.8))
median_handles = []
fit_handles = []
fit_rows = []

for relay in RELAY_ORDER:
    relay_data = (
        curve_df.loc[curve_df['relay'].eq(relay), ['mbps', 'power']]
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
    )
    relay_data = relay_data[relay_data['mbps'] > 0]
    visible = relay_data[relay_data['mbps'].between(0, x_high)]
    sample = visible.sample(n=min(2200, len(visible)), random_state=42)
    boundary = relay_data[relay_data['mbps'] <= LOW_THROUGHPUT_MAX]
    if len(boundary) > 600:
        boundary = boundary.sample(n=600, random_state=42)
    sample = pd.concat([sample, boundary]).loc[lambda x: ~x.index.duplicated()]

    ax.scatter(
        sample['mbps'], sample['power'],
        s=18, marker=PAPER_RELAY_MARKERS[relay],
        color=COLORS[relay], alpha=0.11, linewidths=0, rasterized=True,
    )

    curve = binned_curves[
        binned_curves['relay'].eq(relay)
        & binned_curves['variable'].eq('mbps')
    ].sort_values('x')
    visible_curve = curve[curve['x'].le(x_high)]
    ax.plot(
        visible_curve['x'], visible_curve['power_median'],
        color=COLORS[relay], linewidth=PAPER_LINE_WIDTH, zorder=5,
    )

    rho = correlation_df.loc[
        correlation_df['relay'].eq(relay)
        & correlation_df['variable'].eq('mbps'),
        'spearman_rho',
    ].iloc[0]
    median_handles.append(Line2D(
        [0], [0], color=COLORS[relay], linewidth=PAPER_LINE_WIDTH,
        marker=PAPER_RELAY_MARKERS[relay], markersize=PAPER_MARKER_SIZE,
        markerfacecolor=COLORS[relay], markeredgewidth=0,
        label=fr"{relay} median ($\rho$={rho:.2f})",
    ))

    X = np.log(relay_data[['mbps']].to_numpy())
    y = relay_data['power'].to_numpy()
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    model = LinearRegression().fit(X_train, y_train)
    y_pred = model.predict(X_test)
    slope = model.coef_[0]
    intercept = model.intercept_
    fit_grid = np.linspace(relay_data['mbps'].quantile(0.001), min(relay_data['mbps'].max(), x_high), 400)
    fit_power = intercept + slope * np.log(fit_grid)
    ax.plot(
        fit_grid, fit_power, color=COLORS[relay],
        linestyle=':', linewidth=PAPER_LINE_WIDTH, zorder=6,
    )
    fit_handles.append(Line2D(
        [0], [0], color=COLORS[relay], linestyle=':',
        linewidth=PAPER_LINE_WIDTH, label=f'{relay} ln fit',
    ))
    paper_slope, paper_intercept = PAPER_POWER_MODELS.get(relay, (np.nan, np.nan))
    fit_rows.append({
        'relay': relay,
        'rows': len(relay_data),
        'slope': slope,
        'intercept': intercept,
        'test_r2': r2_score(y_test, y_pred),
        'test_rmse_w': np.sqrt(mean_squared_error(y_test, y_pred)),
        'test_mae_w': mean_absolute_error(y_test, y_pred),
        'paper_slope': paper_slope,
        'paper_intercept': paper_intercept,
        'matches_paper_4dp': (
            round(slope, 4) == paper_slope and round(intercept, 4) == paper_intercept
            if np.isfinite(paper_slope) else np.nan
        ),
    })

ax.set_xlim(0, x_high)
ax.set_ylim(0, PAPER_POWER_MAX)
ax.set_xlabel('Throughput (Mbps)', fontsize=PAPER_LABEL_SIZE, labelpad=10)
ax.set_ylabel('Power (W)', fontsize=PAPER_LABEL_SIZE, labelpad=10)
ax.legend(
    handles=median_handles + fit_handles, loc='lower right', ncol=2,
    frameon=True, fontsize=PAPER_LEGEND_SIZE, handlelength=2.2, columnspacing=1.0,
)
format_paper_axes(ax)
fig.subplots_adjust(left=0.16, right=0.98, bottom=0.19, top=0.98)

png_path, pdf_path = save_paper_figure(
    fig, 'paper_power_vs_throughput_linear_with_log_fits',
)
print(png_path)
print(pdf_path)
display(pd.DataFrame(fit_rows).round(4))
plt.show()